# Step 2a — Create the MySQL Schema and Load Cleaned Data

**Input:** `data/cleaned_telco.csv`
**Output:** `telco_churn.customers` table in MySQL

Requires MySQL running locally and a `.env` file in the project root with
`MYSQL_HOST` / `MYSQL_PORT` / `MYSQL_USER` / `MYSQL_PASSWORD` / `MYSQL_DB`
(see `scripts/utils_db.py`, which this notebook imports).

In [1]:
import os
import sys


def find_project_root(marker="requirements.txt"):
    path = os.getcwd()
    for _ in range(5):
        if os.path.exists(os.path.join(path, marker)):
            return path
        path = os.path.dirname(path)
    raise FileNotFoundError(f"Could not locate project root (looked for {marker})")


PROJECT_ROOT = find_project_root()
sys.path.insert(0, os.path.join(PROJECT_ROOT, "scripts"))
print(f"PROJECT_ROOT = {PROJECT_ROOT}")

PROJECT_ROOT = C:\Users\DEVKUMAR\OneDrive\Desktop\DA_Project


In [2]:
import pandas as pd

from utils_db import get_connection, get_db_name

CLEANED_PATH = os.path.join(PROJECT_ROOT, "data", "cleaned_telco.csv")
SCHEMA_PATH = os.path.join(PROJECT_ROOT, "sql", "01_schema.sql")

COLUMNS = [
    "customerID", "gender", "SeniorCitizen", "Partner", "Dependents", "tenure",
    "PhoneService", "MultipleLines", "InternetService", "OnlineSecurity",
    "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV",
    "StreamingMovies", "Contract", "PaperlessBilling", "PaymentMethod",
    "MonthlyCharges", "TotalCharges", "Churn", "AvgMonthlySpend",
    "NumServices", "ChurnFlag",
]

## Create the schema

Runs `sql/01_schema.sql` — creates the `telco_churn` database and `customers` table (dropping/recreating if they already exist).

In [3]:
conn = get_connection(database=None)
cur = conn.cursor()
with open(SCHEMA_PATH, encoding="utf-8") as f:
    schema_sql = f.read()
for statement in schema_sql.split(";"):
    statement = statement.strip()
    if statement:
        cur.execute(statement)
conn.commit()
cur.close()
conn.close()
print(f"Schema created in database '{get_db_name()}'")

Schema created in database 'telco_churn'


## Load the cleaned data

In [4]:
df = pd.read_csv(CLEANED_PATH)
df = df[COLUMNS]

conn = get_connection()
cur = conn.cursor()
cur.execute("TRUNCATE TABLE customers")

placeholders = ", ".join(["%s"] * len(COLUMNS))
insert_sql = f"INSERT INTO customers ({', '.join(COLUMNS)}) VALUES ({placeholders})"

rows = [tuple(r) for r in df.itertuples(index=False, name=None)]
cur.executemany(insert_sql, rows)
conn.commit()

cur.execute("SELECT COUNT(*) FROM customers")
count = cur.fetchone()[0]
cur.close()
conn.close()

print(f"Loaded {count} rows into customers table (source CSV had {len(df)} rows)")
assert count == len(df), "Row count mismatch after load"

Loaded 7043 rows into customers table (source CSV had 7043 rows)
